# Battery RUL Predictor — Model Comparison (Approach 2)

Compares all five models trained under Approach 2 (train on RW9, predict RW10-RW12 —
the cross-battery generalization test) against the published SRA 2023 baselines.
Loads precomputed predictions from `data/processed/predictions/` (written by
`scripts/precompute_predictions.py`) rather than re-running inference.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from battery_rul.config import PAPER_BASELINE_RMSE, PREDICTIONS_DIR, TEST_BATTERIES
from battery_rul.evaluation.metrics import overestimation_rmse, rmse, underestimation_rmse

MODELS = ["paper_dnn", "upgraded_dnn", "lstm", "attention", "lightgbm"]
MODEL_LABELS = {
    "paper_dnn": "Paper DNN (ours)",
    "upgraded_dnn": "Upgraded DNN (ours)",
    "lstm": "LSTM (ours)",
    "attention": "Attention (ours)",
    "lightgbm": "LightGBM (ours)",
}

predictions = {
    model: {b: pd.read_parquet(PREDICTIONS_DIR / f"{model}_approach2_{b}.parquet") for b in TEST_BATTERIES}
    for model in MODELS
}

## Actual vs Predicted SOH (Paper DNN — the best-performing model)

The models predict SOH, not voltage, so this reproduces the paper's "actual vs
predicted" comparison on the quantity each model actually outputs (see
`CHANGELOG.md` Phase 5 for the same adaptation made in the dashboard).

In [2]:
fig = make_subplots(rows=1, cols=3, subplot_titles=TEST_BATTERIES)
for i, b in enumerate(TEST_BATTERIES):
    df = predictions["paper_dnn"][b]
    fig.add_trace(
        go.Scatter(x=df["absolute_time_raw"], y=df["soh_actual"] * 100, mode="lines", name="Actual"),
        row=1, col=i + 1,
    )
    fig.add_trace(
        go.Scatter(x=df["absolute_time_raw"], y=df["soh_predicted"] * 100, mode="lines", name="Predicted"),
        row=1, col=i + 1,
    )
fig.update_layout(height=400, title="Actual vs Predicted SOH — Paper DNN (Approach 2)")
fig.show()

**Finding:** predicted SOH tracks actual SOH closely on RW10/RW11; RW12 shows the
largest gap, consistent with it being the most degraded, noisiest cell across every
model in this comparison.

## RMSE Training Curves (torch models)

In [3]:
torch_models = ["paper_dnn", "upgraded_dnn", "lstm", "attention"]
fig = make_subplots(rows=2, cols=2, subplot_titles=[MODEL_LABELS[m] for m in torch_models])
for i, model in enumerate(torch_models):
    history = pd.read_parquet(PREDICTIONS_DIR / f"{model}_approach2_history.parquet")
    row, col = i // 2 + 1, i % 2 + 1
    fig.add_trace(go.Scatter(x=history["epoch"], y=history["train_rmse"], name="train", legendgroup=model), row=row, col=col)
    fig.add_trace(go.Scatter(x=history["epoch"], y=history["val_rmse"], name="val", legendgroup=model), row=row, col=col)
fig.update_layout(height=600, title="Training Curves — RMSE per Epoch")
fig.show()
print("LightGBM has no per-epoch curve: boosting rounds aren't epochs (see tree_trainer.fit_lightgbm).")

LightGBM has no per-epoch curve: boosting rounds aren't epochs (see tree_trainer.fit_lightgbm).


**Finding:** `paper_dnn` converges fastest and to the lowest validation RMSE despite
being the smallest network — consistent with the Phase 4/Phase 7 pattern that added
capacity doesn't help on this problem. `upgraded_dnn`'s curve is visibly noisier,
suggesting its extra capacity is fitting noise rather than signal.

## Final RMSE Comparison

In [4]:
rows = list(PAPER_BASELINE_RMSE.items())
for model in MODELS:
    rmses = [
        rmse(df["soh_actual"].to_numpy(), df["soh_predicted"].to_numpy())
        for df in predictions[model].values()
    ]
    rows.append((MODEL_LABELS[model], float(np.mean(rmses))))

comparison = pd.DataFrame(rows, columns=["Model", "Avg RMSE"]).sort_values("Avg RMSE")
comparison["Avg RMSE"] = (comparison["Avg RMSE"] * 100).round(2).astype(str) + "%"
comparison

,Model,Avg RMSE
3,Paper DNN (ours),0.66%
7,LightGBM (ours),0.68%
6,Attention (ours),0.8%
5,LSTM (ours),0.81%
4,Upgraded DNN (ours),1.26%
2,Paper DNN (original),1.49%
0,BLS-RVM,1.55%
1,RNN + LSTM,1.61%


**Finding:** `paper_dnn` (0.66%) wins outright, with `lightgbm` (0.68%) a close
second — strong evidence that the per-row engineered features (rolling mean/std,
dv/dt) already capture the temporal signal that matters, so a model with no notion
of sequence order loses almost nothing. `attention` (0.80%) and `lstm` (0.83%) land
in the middle; `upgraded_dnn` (1.26%) is worst despite being the most expressive
network. All five beat every published SRA 2023 baseline (BLS-RVM 1.55%, RNN+LSTM
1.61%, Paper DNN original 1.49%).

## Overestimation vs Underestimation

In [5]:
safety_rows = []
for model in MODELS:
    actual = np.concatenate([df["soh_actual"].to_numpy() for df in predictions[model].values()])
    predicted = np.concatenate([df["soh_predicted"].to_numpy() for df in predictions[model].values()])
    safety_rows.append(
        (MODEL_LABELS[model], overestimation_rmse(actual, predicted), underestimation_rmse(actual, predicted))
    )

safety_df = pd.DataFrame(safety_rows, columns=["Model", "Overestimation RMSE", "Underestimation RMSE"])
fig = go.Figure()
fig.add_trace(go.Bar(x=safety_df["Model"], y=safety_df["Overestimation RMSE"] * 100, name="Overestimation"))
fig.add_trace(go.Bar(x=safety_df["Model"], y=safety_df["Underestimation RMSE"] * 100, name="Underestimation"))
fig.update_layout(barmode="group", title="Overestimation vs Underestimation RMSE by Model", yaxis_title="RMSE (%)")
fig.show()

**Finding:** for a safety-critical use case like a BMS, underestimating SOH
(prematurely flagging a healthy battery for replacement) is far cheaper than
overestimating it (missing a battery that actually needs replacement). Models whose
overestimation RMSE is lower than their underestimation RMSE are erring on the
conservative, safer side.